# AI Quote Agent
Tests:
1. Tool functions directly
2. Agent multi-step reasoning
3. Quote + invoicing
4. Policy enforcement
5. Eval suite — simple and complex scenarios

In [1]:
import os, sys, json, time, django
from pathlib import Path

notebook_dir = Path.cwd()
python_dir   = notebook_dir.parent
sys.path.insert(0, str(python_dir))

os.environ.setdefault('DJANGO_SETTINGS_MODULE', 'django_app.settings')
django.setup()

from django_app.domain.product_service import get_by_id, list_products
products, total = list_products(page=1, page_size=10)
print(f"Found {len(products)} products in database")
for p in products[:5]:
    print(f"  - {p.name} (ID: {p.id}) - \u20b9{p.price}")

if products:
    test_product_id = str(products[0].id)
    print(f"\nUsing product ID for tests: {test_product_id}")
else:
    test_product_id = None
    print("No products found. Please add products to the database first.")


Found 10 products in database
  - ESP8266 NodeMCU Development Board (ID: 69ba3dd20791de99c0e5f014) - ₹300.0
  - Lithium Polymer (LiPo) Battery (7.4V 1500mAh) (ID: 69ba3dd20791de99c0e5f013) - ₹950.0
  - Logic Level Shifter (Bi-directional) (ID: 69ba3dd20791de99c0e5f012) - ₹150.0
  - RFID RC522 Module Kit (ID: 69ba3dd20791de99c0e5f011) - ₹400.0
  - OLED Display (0.96 inch, I2C) (ID: 69ba3dd20791de99c0e5f010) - ₹350.0

Using product ID for tests: 69ba3dd20791de99c0e5f014


## Part 2: Test Individual Tools

In [2]:
from django_app.domain.quote_agent_service import QuoteAgentService

agent = QuoteAgentService()
tools = agent._build_tools()
get_product_info = next(t for t in tools if t.name == "get_product_info")
check_inventory = next(t for t in tools if t.name == "check_inventory")
calculate_quote = next(t for t in tools if t.name == "calculate_quote")
search_products = next(t for t in tools if t.name == "search_products")
compare_products = next(t for t in tools if t.name == "compare_products")
multi_item_quote = next(t for t in tools if t.name == "multi_item_quote")

W0319 06:26:18.425000 41688 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
c:\Users\Arjun\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [3]:
print("=" * 60)
print("TEST 1: get_product_info")
print("=" * 60)
result = get_product_info.invoke(test_product_id)
print(json.dumps(result, indent=2))

TEST 1: get_product_info
{
  "id": "69ba3dd20791de99c0e5f014",
  "name": "ESP8266 NodeMCU Development Board",
  "brand": "Espressif",
  "category": "Microcontrollers",
  "unit_price": 300.0,
  "quantity_available": 80,
  "description": "Wi-Fi enabled microcontroller for IoT projects, smaller and more affordable than ESP32."
}


In [4]:
if test_product_id:
    print("=" * 60)
    print("TEST 2: check_inventory")
    print("=" * 60)
    result = check_inventory.invoke({"product_id": test_product_id})
    print(json.dumps(result, indent=2))

TEST 2: check_inventory
{
  "product_id": "69ba3dd20791de99c0e5f014",
  "product_name": "ESP8266 NodeMCU Development Board",
  "quantity_available": 80,
  "in_stock": true,
  "status": "Available",
  "reorder_suggested": false
}


In [5]:
if test_product_id:
    print("=" * 60)
    print("TEST 3: calculate_quote — all discount tiers")
    print("=" * 60)
    for qty in [5, 19, 20, 49, 50, 99, 100, 200]:
        r = calculate_quote.invoke({"product_id": test_product_id, "quantity": qty})
        if "error" in r:
            print(f"  qty {qty:>4}: ERROR — {r['error']}")
        else:
            print(
                f"  qty {qty:>4}: \u20b9{r['unit_price']} x {qty} = \u20b9{r['subtotal']}"
                f"  -> {r['discount_pct']}% off -> \u20b9{r['total']}"
                f"  [{r['discount_rule']}]"
            )

TEST 3: calculate_quote — all discount tiers
  qty    5: ₹300.0 x 5 = ₹1500.0  -> 0% off -> ₹1500.0  [No discount]
  qty   19: ₹300.0 x 19 = ₹5700.0  -> 0% off -> ₹5700.0  [No discount]
  qty   20: ₹300.0 x 20 = ₹6000.0  -> 5% off -> ₹5700.0  [5% off for ≥20 units]
  qty   49: ₹300.0 x 49 = ₹14700.0  -> 5% off -> ₹13965.0  [5% off for ≥20 units]
  qty   50: ₹300.0 x 50 = ₹15000.0  -> 10% off -> ₹13500.0  [10% off for ≥50 units]
  qty   99: ₹300.0 x 99 = ₹29700.0  -> 10% off -> ₹26730.0  [10% off for ≥50 units]
  qty  100: ₹300.0 x 100 = ₹30000.0  -> 20% off -> ₹24000.0  [20% off for ≥100 units]
  qty  200: ₹300.0 x 200 = ₹60000.0  -> 20% off -> ₹48000.0  [20% off for ≥100 units]


In [6]:
if products:
    print("=" * 60)
    print("TEST 4: search_products")
    print("=" * 60)
    query   = products[0].name.split()[0]
    results = search_products.invoke(query)
    print(f"Search: '{query}' -> {len(results)} result(s)")
    for r in results:
        if "error" not in r:
            print(f"  - {r['name']} (₹{r['unit_price']}, {r['quantity_available']} in stock)")

TEST 4: search_products
Search: 'ESP8266' -> 2 result(s)
  - ESP8266 NodeMCU Development Board (₹300.0, 80 in stock)
  - NodeMCU ESP8266 Development Board (₹350.0, 30 in stock)


In [7]:
if products:
    print("=" * 60)
    print("TEST 4: compare_products")
    print("=" * 60)
    product_ids = [str(p.id) for p in products[:2]]
    results = compare_products.invoke({"product_ids": product_ids})
    print(f"Compare: {product_ids} -> {len(results)} result(s)")
    for r in results:
        if "error" not in r:
            print(f"  - {r['name']} (₹{r['unit_price']}, {r['quantity_available']} in stock)")

TEST 4: compare_products
Compare: ['69ba3dd20791de99c0e5f014', '69ba3dd20791de99c0e5f013'] -> 2 result(s)
  - ESP8266 NodeMCU Development Board (₹300.0, 80 in stock)
  - Lithium Polymer (LiPo) Battery (7.4V 1500mAh) (₹950.0, 25 in stock)


In [8]:
if products:
    print("=" * 60)
    print("TEST 5: multi item quote")
    print("=" * 60)
    items = [{"product_id": str(p.id), "quantity": 100} for p in products[:4]]
    results = multi_item_quote.invoke({"items": items})
    print(f"Multi-item quote: {len(items)} items")
    if "line_items" in results:
        for item in results["line_items"]:
            print(f"  - {item['product_name']}: {item['quantity']} units @ ₹{item['unit_price']} -> ₹{item['total']} ({item['discount_rule']})")
        print(f"Order total: ₹{results.get('order_total', 0)}")
        print(f"Total savings: ₹{results.get('total_savings', 0)}")
        print(f"Item count: {results.get('item_count', 0)}")
    else:
        print(f"Error: {results}")

TEST 5: multi item quote
Multi-item quote: 4 items
  - ESP8266 NodeMCU Development Board: 100 units @ ₹300.0 -> ₹24000.0 (20% off for ≥100 units)
  - Lithium Polymer (LiPo) Battery (7.4V 1500mAh): 100 units @ ₹950.0 -> ₹76000.0 (20% off for ≥100 units)
  - Logic Level Shifter (Bi-directional): 100 units @ ₹150.0 -> ₹12000.0 (20% off for ≥100 units)
  - RFID RC522 Module Kit: 100 units @ ₹400.0 -> ₹32000.0 (20% off for ≥100 units)
Order total: ₹144000.0
Total savings: ₹36000.0
Item count: 4


## Part 4: Eval Suite — Simple Scenarios
Run **one cell at a time** to avoid rate limits, or let the built-in `time.sleep(15)` handle it.

In [25]:
print(f"Agent initialized with {len(agent.tools)} tools")
print(f"Tools: {[t.name for t in agent.tools]}")

Agent initialized with 6 tools
Tools: ['search_products', 'get_product_info', 'check_inventory', 'calculate_quote', 'compare_products', 'multi_item_quote']


In [ ]:
if products:
    time.sleep(15)
    product_name = products[0].name
    query = f"How many {product_name} do we have in stock?"
    print(f"Query: {query}")

    result = agent.process_request(query)
    response = result.get('response', '')
    if isinstance(response, list):
        response = ' '.join(
            x.get('text', str(x)) if isinstance(x, dict) else str(x)
            for x in response
        )
    else:
        response = str(response)
    print(f"Response: {response[:300]}")
    print(f"Tools:    {[tc['tool'] for tc in result.get('tool_calls', [])]}")
    expected = ["stock", "available", "quantity"]
    found    = [kw for kw in expected if kw.lower() in response.lower()]
    print(f"Keywords found: {found} / {expected}")


In [ ]:
if products:
    time.sleep(15)
    product_name = products[0].name
    query = f"I need 30 units of {product_name}, what's the total cost?"
    print(f"Query: {query}")

    result = agent.process_request(query)
    response = result.get('response', '')
    if isinstance(response, list):
        response = ' '.join(
            x.get('text', str(x)) if isinstance(x, dict) else str(x)
            for x in response
        )
    else:
        response = str(response)
    print(f"Response: {response[:300]}")
    print(f"Tools:    {[tc['tool'] for tc in result.get('tool_calls', [])]}")
    if result.get('quote'):
        print(f"Quote:    {json.dumps(result['quote'], indent=2)}")


Query: I need 30 units of ESP8266 NodeMCU Development Board, what's the total cost?
[agent] RAG retrieval failed: Expected metadata value to be a str, int, float, bool, SparseVector, or None, got ['PRODUCT MANUAL - Inventory Management System\n==========================================\n\nVersion: 2.0\nLast Updated: March 2026\n\nSECTION 1: OVERVIEW\n-------------------\nThe Inventory Management System is designed to help businesses track, manage, and optimize their product inventory. This system supports multiple product categories, brands, and provides AI-powered assistance for inventory operations. SECTION 2: PRODUCT SPECIFICATIONS\n-----------------------------------\nEach product in the system contains the following fields:\n- Product Name: Unique identifier for the product\n- Brand: Manufacturer or supplier brand\n- Category: Classification group (Electronics, Furniture, Clothing, etc.)\n- Description: Detailed product information\n- Price: Unit price in INR (Indian Rupees)\n- Qu

[03/19/26 03:53:15] INFO     HTTP Request: POST                                                     ]8;id=249119;file://c:\Users\Arjun\anaconda3\Lib\site-packages\httpx\_client.py\_client.py]8;;\:]8;id=164932;file://c:\Users\Arjun\anaconda3\Lib\site-packages\httpx\_client.py#1025\1025]8;;\
                             https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-fla                
                             sh:generateContent "HTTP/1.1 200 OK"                                                  

[03/19/26 03:53:18] INFO     HTTP Request: POST                                                     ]8;id=327831;file://c:\Users\Arjun\anaconda3\Lib\site-packages\httpx\_client.py\_client.py]8;;\:]8;id=595138;file://c:\Users\Arjun\anaconda3\Lib\site-packages\httpx\_client.py#1025\1025]8;;\
                             https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-fla                
                             sh:generateContent "HTTP/1.1 200 OK"                                                  

[03/19/26 03:53:20] INFO     HTTP Request: POST                                                     ]8;id=581907;file://c:\Users\Arjun\anaconda3\Lib\site-packages\httpx\_client.py\_client.py]8;;\:]8;id=831308;file://c:\Users\Arjun\anaconda3\Lib\site-packages\httpx\_client.py#1025\1025]8;;\
                             https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-fla                
                             sh:generateContent "HTTP/1.1 200 OK"                                                  

Response: The total cost for 30 units of ESP8266 NodeMCU Development Board is 8550.
Tools:    ['search_products', 'calculate_quote']
Quote:    {
  "product_id": "69ba3dd20791de99c0e5f014",
  "product_name": "ESP8266 NodeMCU Development Board",
  "quantity": 30,
  "unit_price": 300.0,
  "discount_pct": 5,
  "discount_amount": 450.0,
  "subtotal": 9000.0,
  "total": 8550.0,
  "discount_rule": "5% off for quantities \u2265 20"
}


## Part 5: Eval Suite — Complex Scenarios

In [10]:
if products:
    time.sleep(15)
    product_name = products[0].name
    query = f"I need 60 {product_name} for a school project. Can I get a deal? What's the total?"
    print(f"Scenario: Multi-step Search + Quote")
    print(f"Query: {query}")

    result = agent.process_request(query)
    response = result.get('response', '')
    if isinstance(response, list):
        response = ' '.join(
            x.get('text', str(x)) if isinstance(x, dict) else str(x)
            for x in response
        )
    else:
        response = str(response)
    print(f"Response: {response[:400]}")
    print(f"Tools:    {[tc['tool'] for tc in result.get('tool_calls', [])]}")
    if result.get('quote'):
        q = result['quote']
        print(f"Quote:    {q['quantity']} units @ \u20b9{q['unit_price']} -> {q['discount_pct']}% off -> \u20b9{q['total']}")
        status = "\u2713 PASS" if q['discount_pct'] == 10 else f"\u2717 FAIL — expected 10%, got {q['discount_pct']}%"
        print(f"Discount: {status}")


Scenario: Multi-step Search + Quote
Query: I need 60 ESP8266 NodeMCU Development Board for a school project. Can I get a deal? What's the total?
[agent] RAG retrieval failed: Expected metadata value to be a str, int, float, bool, SparseVector, or None, got ['PRODUCT MANUAL - Inventory Management System\n==========================================\n\nVersion: 2.0\nLast Updated: March 2026\n\nSECTION 1: OVERVIEW\n-------------------\nThe Inventory Management System is designed to help businesses track, manage, and optimize their product inventory. This system supports multiple product categories, brands, and provides AI-powered assistance for inventory operations. SECTION 2: PRODUCT SPECIFICATIONS\n-----------------------------------\nEach product in the system contains the following fields:\n- Product Name: Unique identifier for the product\n- Brand: Manufacturer or supplier brand\n- Category: Classification group (Electronics, Furniture, Clothing, etc.)\n- Description: Detailed product 

[03/19/26 03:53:44] INFO     HTTP Request: POST                                                     ]8;id=479103;file://c:\Users\Arjun\anaconda3\Lib\site-packages\httpx\_client.py\_client.py]8;;\:]8;id=591330;file://c:\Users\Arjun\anaconda3\Lib\site-packages\httpx\_client.py#1025\1025]8;;\
                             https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-fla                
                             sh:generateContent "HTTP/1.1 200 OK"                                                  

[03/19/26 03:53:46] INFO     HTTP Request: POST                                                     ]8;id=856630;file://c:\Users\Arjun\anaconda3\Lib\site-packages\httpx\_client.py\_client.py]8;;\:]8;id=370377;file://c:\Users\Arjun\anaconda3\Lib\site-packages\httpx\_client.py#1025\1025]8;;\
                             https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-fla                
                             sh:generateContent "HTTP/1.1 200 OK"                                                  

[03/19/26 03:53:49] INFO     HTTP Request: POST                                                     ]8;id=433228;file://c:\Users\Arjun\anaconda3\Lib\site-packages\httpx\_client.py\_client.py]8;;\:]8;id=177933;file://c:\Users\Arjun\anaconda3\Lib\site-packages\httpx\_client.py#1025\1025]8;;\
                             https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-fla                
                             sh:generateContent "HTTP/1.1 200 OK"                                                  

[03/19/26 03:53:52] INFO     HTTP Request: POST                                                     ]8;id=397530;file://c:\Users\Arjun\anaconda3\Lib\site-packages\httpx\_client.py\_client.py]8;;\:]8;id=741549;file://c:\Users\Arjun\anaconda3\Lib\site-packages\httpx\_client.py#1025\1025]8;;\
                             https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-fla                
                             sh:generateContent "HTTP/1.1 200 OK"                                                  

Response: Yes, you can get a deal! You'll receive a 10% discount for ordering 60 units, which saves you 1800. Your total will be 16200.
Tools:    ['search_products', 'calculate_quote', 'calculate_quote']
Quote:    60 units @ ₹300.0 -> 10% off -> ₹16200.0
Discount: ✓ PASS


In [11]:
if products:
    time.sleep(15)
    product_name = products[0].name
    query = f"We need 200 {product_name} for our store. Do we have enough stock and what's the bulk price?"
    print(f"Scenario: Bulk order with inventory check")
    print(f"Query: {query}")

    result = agent.process_request(query)
    response = result.get('response', '')
    if isinstance(response, list):
        response = ' '.join(
            x.get('text', str(x)) if isinstance(x, dict) else str(x)
            for x in response
        )
    else:
        response = str(response)
    print(f"Response: {response[:400]}")
    print(f"Tools:    {[tc['tool'] for tc in result.get('tool_calls', [])]}")
    if result.get('quote'):
        q = result['quote']
        print(f"Quote:    {q['quantity']} units, {q['discount_pct']}% off, total \u20b9{q['total']}")
        status = "\u2713 PASS" if q['discount_pct'] == 20 else f"\u2717 FAIL — expected 20%, got {q['discount_pct']}%"
        print(f"Discount: {status}")


Scenario: Bulk order with inventory check
Query: We need 200 ESP8266 NodeMCU Development Board for our store. Do we have enough stock and what's the bulk price?
[agent] RAG retrieval failed: Expected metadata value to be a str, int, float, bool, SparseVector, or None, got ['PRODUCT MANUAL - Inventory Management System\n==========================================\n\nVersion: 2.0\nLast Updated: March 2026\n\nSECTION 1: OVERVIEW\n-------------------\nThe Inventory Management System is designed to help businesses track, manage, and optimize their product inventory. This system supports multiple product categories, brands, and provides AI-powered assistance for inventory operations. SECTION 2: PRODUCT SPECIFICATIONS\n-----------------------------------\nEach product in the system contains the following fields:\n- Product Name: Unique identifier for the product\n- Brand: Manufacturer or supplier brand\n- Category: Classification group (Electronics, Furniture, Clothing, etc.)\n- Description: D

[03/19/26 03:54:23] INFO     HTTP Request: POST                                                     ]8;id=957772;file://c:\Users\Arjun\anaconda3\Lib\site-packages\httpx\_client.py\_client.py]8;;\:]8;id=712814;file://c:\Users\Arjun\anaconda3\Lib\site-packages\httpx\_client.py#1025\1025]8;;\
                             https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-fla                
                             sh:generateContent "HTTP/1.1 200 OK"                                                  

[03/19/26 03:54:25] INFO     HTTP Request: POST                                                     ]8;id=127396;file://c:\Users\Arjun\anaconda3\Lib\site-packages\httpx\_client.py\_client.py]8;;\:]8;id=548185;file://c:\Users\Arjun\anaconda3\Lib\site-packages\httpx\_client.py#1025\1025]8;;\
                             https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-fla                
                             sh:generateContent "HTTP/1.1 200 OK"                                                  

[03/19/26 03:54:27] INFO     HTTP Request: POST                                                     ]8;id=427994;file://c:\Users\Arjun\anaconda3\Lib\site-packages\httpx\_client.py\_client.py]8;;\:]8;id=19309;file://c:\Users\Arjun\anaconda3\Lib\site-packages\httpx\_client.py#1025\1025]8;;\
                             https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-fla                
                             sh:generateContent "HTTP/1.1 200 OK"                                                  

Response: We do not have 200 units of the ESP8266 NodeMCU Development Board in stock. We only have 80 units available.

For a bulk order of 200 units, the price would be 48000 with a 20% discount. The unit price would be 240.
Tools:    ['search_products', 'calculate_quote']
Quote:    200 units, 20% off, total ₹48000.0
Discount: ✓ PASS


In [7]:
if len(products) >= 2:
    time.sleep(15)
    p1, p2 = products[4].name, products[5].name
    query = f"Compare prices of {p1} and {p2}"
    print(f"Scenario: Product comparison")
    print(f"Query: {query}")

    result = agent.process_request(query)
    response = result.get('response', '')
    if isinstance(response, list):
        response = ' '.join(
            x.get('text', str(x)) if isinstance(x, dict) else str(x)
            for x in response
        )
    else:
        response = str(response)
    print(f"Response: {response[:400]}")
    print(f"Tools:    {[tc['tool'] for tc in result.get('tool_calls', [])]}")


Scenario: Product comparison
Query: Compare prices of OLED Display (0.96 inch, I2C) and Micro SD Card (32GB, Class 10)


[03/19/26 04:20:13] INFO     Anonymized telemetry enabled. See                                        ]8;id=958952;file://c:\Users\Arjun\anaconda3\Lib\site-packages\chromadb\telemetry\product\posthog.py\posthog.py]8;;\:]8;id=112268;file://c:\Users\Arjun\anaconda3\Lib\site-packages\chromadb\telemetry\product\posthog.py#22\22]8;;\
                             https://docs.trychroma.com/telemetry for more information.                            

[03/19/26 04:20:18] INFO     HTTP Request: POST                                                     ]8;id=89229;file://c:\Users\Arjun\anaconda3\Lib\site-packages\httpx\_client.py\_client.py]8;;\:]8;id=674530;file://c:\Users\Arjun\anaconda3\Lib\site-packages\httpx\_client.py#1025\1025]8;;\
                             https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-fla                
                             sh:generateContent "HTTP/1.1 200 OK"                                                  

[03/19/26 04:20:20] INFO     HTTP Request: POST                                                     ]8;id=522075;file://c:\Users\Arjun\anaconda3\Lib\site-packages\httpx\_client.py\_client.py]8;;\:]8;id=165548;file://c:\Users\Arjun\anaconda3\Lib\site-packages\httpx\_client.py#1025\1025]8;;\
                             https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-fla                
                             sh:generateContent "HTTP/1.1 200 OK"                                                  

[03/19/26 04:20:22] INFO     HTTP Request: POST                                                     ]8;id=434998;file://c:\Users\Arjun\anaconda3\Lib\site-packages\httpx\_client.py\_client.py]8;;\:]8;id=898643;file://c:\Users\Arjun\anaconda3\Lib\site-packages\httpx\_client.py#1025\1025]8;;\
                             https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-fla                
                             sh:generateContent "HTTP/1.1 200 OK"                                                  

Response: The OLED Display (0.96 inch, I2C) costs 350, while the Micro SD Card (32GB, Class 10) costs 450. The Micro SD Card is more expensive.
Tools:    ['search_products', 'search_products']


In [8]:
if products:
    time.sleep(15)
    p1, p2 = products[0].name, products[1].name
    query = f"What quantity do I need to buy to get a 20% discount on {p1} and {p2}?"
    print(f"Scenario: Discount threshold query")
    print(f"Query: {query}")

    result = agent.process_request(query)
    response = result.get('response', '')
    if isinstance(response, list):
        response = ' '.join(
            x.get('text', str(x)) if isinstance(x, dict) else str(x)
            for x in response
        )
    else:
        response = str(response)
    print(f"Response: {response[:400]}")
    print(f"Tools:    {[tc['tool'] for tc in result.get('tool_calls', [])]}")


Scenario: Discount threshold query
Query: What quantity do I need to buy to get a 20% discount on ESP8266 NodeMCU Development Board and Lithium Polymer (LiPo) Battery (7.4V 1500mAh)?


[03/19/26 04:20:58] INFO     HTTP Request: POST                                                     ]8;id=99697;file://c:\Users\Arjun\anaconda3\Lib\site-packages\httpx\_client.py\_client.py]8;;\:]8;id=209970;file://c:\Users\Arjun\anaconda3\Lib\site-packages\httpx\_client.py#1025\1025]8;;\
                             https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-fla                
                             sh:generateContent "HTTP/1.1 200 OK"                                                  

Response: To get a 20% discount on the ESP8266 NodeMCU Development Board and the Lithium Polymer (LiPo) Battery (7.4V 1500mAh), you would need to buy 100 or more units of each product.
Tools:    []


## Part 6: Quote Generation and Invoicing

In [8]:
quote_data = {
    "product_id":      "product_001",
    "product_name":    "Lego Castle",
    "quantity":        75,
    "unit_price":      79.99,
    "discount_pct":    10,
    "discount_amount": round(79.99 * 75 * 0.10, 2),
    "subtotal":        round(79.99 * 75, 2),
    "total":           round(79.99 * 75 * 0.90, 2),
}

print("=" * 60)
print("Invoice Generation")
print("=" * 60)

invoice = agent.generate_quote_invoice(quote_data)
print(json.dumps(invoice, indent=2))

assert invoice['invoice_id'].startswith('QT-'), "Invoice ID must start with QT-"
assert invoice['total_amount'] == quote_data['total'], "Invoice total must match quote total"
print("\n\u2713 Invoice structure correct")


Invoice Generation
{
  "invoice_id": "QT-20260319040455",
  "timestamp": "2026-03-19T04:04:55.139487",
  "items": [
    {
      "product_id": "product_001",
      "product_name": "Lego Castle",
      "quantity": 75,
      "unit_price": 79.99,
      "discount_pct": 10,
      "discount_amount": 599.93,
      "subtotal": 5999.25,
      "total": 5399.32
    }
  ],
  "total_amount": 5399.32,
  "total_discount": 599.93
}

✓ Invoice structure correct
